# 04 — Evaluation et benchmark

Compare notre modele fine-tune vs les baselines (NAFNet vanilla, Real-ESRGAN) sur le test set.

**Metriques :** PSNR, SSIM, OCR accuracy (EasyOCR)

**Runtime : GPU T4 recommande, ~30 min**

In [ ]:
!pip install -q torch torchvision pillow scikit-image easyocr realesrgan basicsr gfpgan tqdm matplotlib pandas

In [ ]:
import torch
import numpy as np
import cv2
import json
import time
from PIL import Image
from tqdm import tqdm
from skimage.metrics import peak_signal_noise_ratio as calc_psnr
from skimage.metrics import structural_similarity as calc_ssim
import easyocr
import matplotlib.pyplot as plt
import pandas as pd

# OCR reader
reader = easyocr.Reader(['fr', 'en'], gpu=torch.cuda.is_available())

# Charger le test set
with open('dataset/test.json') as f:
    test_pairs = json.load(f)
print(f"Test set : {len(test_pairs)} paires")

## 1. Charger les modeles

In [ ]:
import sys
sys.path.insert(0, 'NAFNet')
from basicsr.models.archs.NAFNet_arch import NAFNet
from realesrgan import RealESRGANer
from basicsr.archs.rrdbnet_arch import RRDBNet

# --- Notre modele (fine-tune) ---
checkpoint = torch.load('checkpoints/nafnet_plates_fr_full.pth', map_location='cpu')
config = checkpoint['config']
our_model = NAFNet(**config)
our_model.load_state_dict(checkpoint['model_state_dict'])
our_model = our_model.cuda().eval()
print(f"Notre modele charge (best PSNR: {checkpoint['best_val_psnr']:.2f} dB)")

# --- NAFNet vanilla (pre-entraine GoPro, pas fine-tune) ---
nafnet_vanilla = NAFNet(**config)
state = torch.load('checkpoints/NAFNet-GoPro-width64.pth', map_location='cpu')
if 'params' in state: state = state['params']
nafnet_vanilla.load_state_dict(state, strict=False)
nafnet_vanilla = nafnet_vanilla.cuda().eval()
print("NAFNet vanilla charge")

# --- Real-ESRGAN ---
!wget -q https://github.com/xinntao/Real-ESRGAN/releases/download/v0.1.0/RealESRGAN_x4plus.pth -P checkpoints/ 2>/dev/null
esrgan_model = RRDBNet(num_in_ch=3, num_out_ch=3, num_feat=64, num_block=23, num_grow_ch=32, scale=4)
esrgan = RealESRGANer(
    scale=4, model_path='checkpoints/RealESRGAN_x4plus.pth',
    model=esrgan_model, tile=0, half=True
)
print("Real-ESRGAN charge")

## 2. Fonctions d'inference

In [ ]:
from torchvision import transforms

to_tensor = transforms.ToTensor()

def infer_nafnet(model, img_bgr):
    """Inference NAFNet sur une image BGR."""
    img_rgb = cv2.cvtColor(img_bgr, cv2.COLOR_BGR2RGB)
    tensor = to_tensor(img_rgb).unsqueeze(0).cuda()
    with torch.no_grad():
        out = model(tensor)
    out = out.squeeze(0).cpu().numpy().transpose(1, 2, 0).clip(0, 1)
    return (out * 255).astype(np.uint8)

def infer_esrgan(img_bgr):
    """Inference Real-ESRGAN sur une image BGR."""
    output, _ = esrgan.enhance(img_bgr, outscale=4)
    # Resize back pour comparer equitablement
    return cv2.resize(output, (img_bgr.shape[1], img_bgr.shape[0]), interpolation=cv2.INTER_LANCZOS4)

def ocr_accuracy(img_rgb, ground_truth):
    """Mesure si l'OCR lit correctement la plaque."""
    results = reader.readtext(img_rgb, detail=0)
    detected = ''.join(results).replace(' ', '').replace('-', '').upper()
    gt = ground_truth.replace('-', '').upper()
    # Compter les caracteres corrects
    correct = sum(1 for a, b in zip(detected, gt) if a == b)
    return correct / max(len(gt), 1), detected

## 3. Benchmark complet

In [ ]:
N_TEST = min(500, len(test_pairs))  # 500 images pour le benchmark
sample = test_pairs[:N_TEST]

results = {'Degradee': [], 'Notre modele': [], 'NAFNet vanilla': [], 'Real-ESRGAN': []}

for pair in tqdm(sample, desc='Benchmark'):
    clean_bgr = cv2.imread(f"dataset/clean/{pair['clean']}")
    deg_bgr = cv2.imread(f"dataset/degraded/{pair['degraded']}")
    clean_rgb = cv2.cvtColor(clean_bgr, cv2.COLOR_BGR2RGB)
    gt_text = pair['text']

    for name, restored_rgb in [
        ('Degradee', cv2.cvtColor(deg_bgr, cv2.COLOR_BGR2RGB)),
        ('Notre modele', infer_nafnet(our_model, deg_bgr)),
        ('NAFNet vanilla', infer_nafnet(nafnet_vanilla, deg_bgr)),
        ('Real-ESRGAN', cv2.cvtColor(infer_esrgan(deg_bgr), cv2.COLOR_BGR2RGB)),
    ]:
        p = calc_psnr(clean_rgb, restored_rgb, data_range=255)
        s = calc_ssim(clean_rgb, restored_rgb, channel_axis=2, data_range=255)
        ocr_acc, ocr_text = ocr_accuracy(restored_rgb, gt_text)
        results[name].append({'psnr': p, 'ssim': s, 'ocr_acc': ocr_acc, 'ocr_text': ocr_text})

print("Benchmark termine")

## 4. Resultats

In [ ]:
# Tableau recap
summary = []
for name, data in results.items():
    summary.append({
        'Modele': name,
        'PSNR (dB)': f"{np.mean([d['psnr'] for d in data]):.2f}",
        'SSIM': f"{np.mean([d['ssim'] for d in data]):.3f}",
        'OCR accuracy': f"{np.mean([d['ocr_acc'] for d in data])*100:.1f}%",
    })

df = pd.DataFrame(summary)
print("\n" + "="*60)
print("RESULTATS BENCHMARK")
print("="*60)
print(df.to_string(index=False))
print("="*60)

In [ ]:
# Graphique comparatif
models = list(results.keys())
psnrs = [np.mean([d['psnr'] for d in results[m]]) for m in models]
ocrs = [np.mean([d['ocr_acc'] for d in results[m]]) * 100 for m in models]
colors = ['#ef4444', '#10b981', '#6366f1', '#f59e0b']

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 5))

ax1.barh(models, psnrs, color=colors)
ax1.set_xlabel('PSNR (dB)')
ax1.set_title('Qualite de restauration (PSNR)')
for i, v in enumerate(psnrs):
    ax1.text(v + 0.3, i, f'{v:.1f}', va='center', fontweight='bold')

ax2.barh(models, ocrs, color=colors)
ax2.set_xlabel('OCR accuracy (%)')
ax2.set_title('Lisibilite OCR (EasyOCR)')
for i, v in enumerate(ocrs):
    ax2.text(v + 1, i, f'{v:.0f}%', va='center', fontweight='bold')

plt.suptitle('Benchmark — Notre modele vs baselines', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.savefig('benchmark_comparison.png', dpi=150)
plt.show()

## 5. Exemples visuels

In [ ]:
# 6 exemples avant/apres
import random
samples = random.sample(test_pairs[:N_TEST], 6)

fig, axes = plt.subplots(6, 4, figsize=(18, 22))
cols = ['Degradee', 'Notre modele', 'NAFNet vanilla', 'Verite terrain']

for i, pair in enumerate(samples):
    clean_bgr = cv2.imread(f"dataset/clean/{pair['clean']}")
    deg_bgr = cv2.imread(f"dataset/degraded/{pair['degraded']}")

    imgs = [
        cv2.cvtColor(deg_bgr, cv2.COLOR_BGR2RGB),
        infer_nafnet(our_model, deg_bgr),
        infer_nafnet(nafnet_vanilla, deg_bgr),
        cv2.cvtColor(clean_bgr, cv2.COLOR_BGR2RGB),
    ]

    for j, (img, col) in enumerate(zip(imgs, cols)):
        axes[i][j].imshow(img)
        if i == 0:
            axes[i][j].set_title(col, fontsize=11, fontweight='bold')
        axes[i][j].axis('off')

    # Annoter avec le texte de la plaque
    axes[i][0].set_ylabel(f"{pair['text']}\n({pair['degradation']})", fontsize=9, rotation=0, labelpad=80)

plt.suptitle('Comparaison visuelle — Degradee → Notre modele → NAFNet vanilla → Ground truth',
             fontsize=14, fontweight='bold')
plt.tight_layout()
plt.savefig('visual_comparison.png', dpi=150, bbox_inches='tight')
plt.show()

## 6. Conclusion

### Resultats cles

| Modele | PSNR | SSIM | OCR accuracy |
|--------|------|------|--------------|
| Image degradee | ... | ... | ... |
| Real-ESRGAN | ... | ... | ... |
| NAFNet (vanilla GoPro) | ... | ... | ... |
| **Notre modele (fine-tune FR)** | **...** | **...** | **...** |

### Ce qui rend ce projet interessant
1. **Dataset original** : premier dataset synthetique de plaques francaises SIV
2. **Fine-tuning** : NAFNet adapte au domaine specifique des plaques
3. **Evaluation OCR** : on ne mesure pas juste la qualite visuelle, mais la lisibilite reelle
4. **Deployable** : modele integre dans un site web avec demo interactive